# ⚽ Futbol Mundial — Tablero de predicciones del Mundial 2026

Predicciones partido a partido y simulación completa del Mundial usando **Transfermarkt** (valor de plantel, lesionados, stats por jugador) + **Elo histórico** (49.000 partidos desde 1872), con la llave oficial de FIFA.

**Cómo usarlo:** ejecutá las celdas en orden con ▶. Las celdas 1 y 2 son la base; después corré los análisis que quieras. Cada celda dice cuánto tarda.

In [ ]:
# 1. Descargar el proyecto y levantar la API de Transfermarkt local (~2 min la primera vez)
# Corre la API dentro del propio Colab: incluye el fix para selecciones nacionales
# y es mucho más rápida que la instancia pública.
import os, subprocess, time
import requests as rq

if not os.path.exists('/content/Futbol-mundial'):
    !git clone -q -b claude/laughing-ritchie-f4b202 https://github.com/GustaPardo/Futbol-mundial.git /content/Futbol-mundial
else:
    !git -C /content/Futbol-mundial pull -q
%cd /content/Futbol-mundial/predicciones
!pip install -q requests

def api_viva():
    try:
        return rq.get('http://localhost:8000/docs', timeout=3).status_code == 200
    except Exception:
        return False

if not api_viva():
    print('Instalando dependencias de la API...')
    !pip install -q -r ../transfermarkt-api/requirements.txt
    env = dict(os.environ, PYTHONPATH='/content/Futbol-mundial/transfermarkt-api')
    subprocess.Popen(['python', '/content/Futbol-mundial/transfermarkt-api/app/main.py'],
                     env=env, stdout=open('/tmp/api.log', 'w'), stderr=subprocess.STDOUT)
    for _ in range(30):
        if api_viva():
            break
        time.sleep(2)

os.environ['TM_API_URL'] = 'http://localhost:8000'
os.environ['TM_API_PAUSA'] = '0.4'   # pausa entre requests, para no abusar de Transfermarkt
print('API local:', 'OK ✅' if api_viva() else '✘ falló — revisá con: !tail -30 /tmp/api.log')

In [ ]:
# 2. Planteles reales de Transfermarkt para los 48 mundialistas
#
# MODO "rapido"   → valor de mercado + edad + descuento de lesionados (~2 min)
# MODO "completo" → además pondera a cada jugador por nivel de competencia
#                   (Champions/ligas top), minutos y goles+asistencias de la
#                   última temporada, y habilita el goleador con datos reales
#                   (~15-20 min; tiene caché: si se corta, re-ejecutá y retoma solo)
MODO = "completo"

flag = "--con-stats" if MODO == "completo" else ""
!python generar_scores.py {flag}

In [ ]:
# 3. Simular el Mundial completo 50.000 veces (~1 min)
# Campeón, finalista y goles esperados por equipo, con la llave oficial FIFA,
# Elo dinámico y localía de los anfitriones.
!python simular_mundial.py -n 50000

In [ ]:
# 4. Análisis grupo por grupo (~30 seg)
# Probabilidad de cada equipo de salir 1°, 2° y de clasificar a 16avos.
!python analizar.py grupos -n 20000

In [ ]:
# 5. Radiografía de un equipo (~30 seg)
# Hasta qué ronda llega, quién lo elimina en cada instancia, y su grupo con ratings.
# Nombres en inglés: "Portugal", "Germany", "Iraq", "Argentina", "Mexico"...
EQUIPO = "Portugal"

!python analizar.py equipo "{EQUIPO}" -n 20000

In [ ]:
# 6. Goleador del Mundial, jugador por jugador (~1 min)
# Reparte los goles esperados de cada selección entre sus jugadores reales.
# Con MODO "completo" en la celda 2 usa los goles/minutos reales de la última
# temporada de cada jugador; si no, estima por posición y valor de mercado.
!python analizar.py goleador -n 20000 --top 20

In [ ]:
# 7. Predicción de un partido suelto (~15 seg)
equipo_a = "Iraq"
equipo_b = "Uzbekistan"
local = ""   # "A" si el primero juega de local, "B" si el segundo, "" neutral

extra = f'--local {local}' if local else ''
!python predictor.py "{equipo_a}" "{equipo_b}" {extra}

### Cómo funciona

- **Fuerza de cada equipo**: 50% rating Elo histórico (calculado desde 49.000 partidos internacionales) + 50% valor del plantel de Transfermarkt, con lesionados descontados y, en modo completo, cada jugador ponderado por el nivel de las competencias donde jugó (Champions y top-5 ligas al máximo), sus minutos y su producción.
- **Modelo de goles Poisson calibrado** con 11.600 partidos reales (2010–2022) y validado sobre 3.500 partidos de 2023+ que nunca vio: **60,7% de acierto** en victoria/empate/derrota (baseline 47%).
- **Simulador**: fixture real de grupos, los 8 mejores terceros con las restricciones de FIFA, la llave oficial (partidos 73–104), Elo dinámico durante el torneo, alargue y penales.

Metodología completa y limitaciones: [`docs/ANALISIS.md`](https://github.com/GustaPardo/Futbol-mundial/blob/claude/laughing-ritchie-f4b202/docs/ANALISIS.md).